# Notebook 09 — Full Results Comparison

Aggregates results from all benchmark notebooks (02–08) and compares
them side-by-side with the paper's Tables 4 (RMSE), 5 (MAD), 6 (CoD).

**Run notebooks 02–08 first** to generate the JSON result files.

### Metric Definitions (matching paper normalisation)

| Metric | Formula | Notes |
|--------|---------|-------|
| RMSE | per-feature RMSE / (max−min), averaged over missing features | Matches Table 4 closely |
| MAD  | per-feature MAD  / (max−min), averaged over missing features | Paper's exact normalisation is undocumented; absolute values differ ~3×, but method *rankings* are preserved |
| CoD  | 1 − SS_res(missing) / SS_tot(all cells) | Matches Table 6 closely |

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Load All Saved Results

In [ ]:
from miga.paper_results import DATASETS, TABLE4_RMSE, TABLE5_MAD, TABLE6_COD

NB_META = {
    "Iris":      "02", "Wine": "03", "Glass": "04",
    "Haberman":  "05", "Wholesale": "06",
    "Cardio":    "07", "Adult": "08",
}

our_results = {}
for ds, num in NB_META.items():
    path = os.path.join(RESULTS_DIR, f"{num}_{ds}_results.json")
    if os.path.exists(path):
        with open(path) as f:
            our_results[ds] = {int(k): v for k, v in json.load(f).items()}
        print(f"  Loaded: {path}")
    else:
        print(f"  Missing: {path}  (run notebook {num} first)")

## 2. RMSE — Our MIGA vs. Paper Table 4

In [ ]:
print("\n=== RMSE Comparison (Our MIGA vs. Paper MIGA) ===")
print(f"{'Dataset':<12} {'pct':>5} {'Ours':>10} {'Paper':>10} {'Δ':>10} {'Better?':>9}")
print("-" * 60)

for ds in DATASETS:
    if ds not in our_results:
        continue
    for pct in PERCENTAGES:
        our  = our_results[ds][pct]["rmse"]
        pap  = TABLE4_RMSE[ds][pct]["MIGA"]
        diff = our - pap
        better = "✓" if our <= pap else "✗"
        print(f"{ds:<12} {pct:>5}% {our:>10.4f} {pap:>10.4f} {diff:>+10.4f} {better:>9}")

## 3. MAD — Our MIGA vs. Paper Table 5

In [ ]:
print("\n=== MAD Comparison (Our MIGA vs. Paper MIGA) ===")
print(f"{'Dataset':<12} {'pct':>5} {'Ours':>10} {'Paper':>10} {'Δ':>10} {'Better?':>9}")
print("-" * 60)

for ds in DATASETS:
    if ds not in our_results:
        continue
    for pct in PERCENTAGES:
        our  = our_results[ds][pct]["mad"]
        pap  = TABLE5_MAD[ds][pct]["MIGA"]
        diff = our - pap
        better = "✓" if our <= pap else "✗"
        print(f"{ds:<12} {pct:>5}% {our:>10.4f} {pap:>10.4f} {diff:>+10.4f} {better:>9}")

## 4. CoD — Our MIGA vs. Paper Table 6

In [ ]:
print("\n=== CoD Comparison (Our MIGA vs. Paper MIGA) ===")
print(f"{'Dataset':<12} {'pct':>5} {'Ours':>10} {'Paper':>10} {'Δ':>10} {'Better?':>9}")
print("-" * 60)

for ds in DATASETS:
    if ds not in our_results:
        continue
    for pct in PERCENTAGES:
        our  = our_results[ds][pct]["cod"]
        pap  = TABLE6_COD[ds][pct]["MIGA"]
        diff = our - pap
        better = "✓" if our >= pap else "✗"
        print(f"{ds:<12} {pct:>5}% {our:>10.4f} {pap:>10.4f} {diff:>+10.4f} {better:>9}")

## 5. Ratio Summary — Our MIGA / Paper MIGA

Ratio = Our RMSE / Paper RMSE.
Values **< 1.0** (★) mean we beat the paper.
Values **1.0–1.2** are excellent for a reimplementation.
Ratios **> 2.5** indicate a known algorithmic limitation.

In [ ]:
print("=== RMSE: Our MIGA / Paper MIGA  (ratio per dataset × missing %) ===")
print()
print(f"  {'Dataset':<12}  {'30%':>9}  {'40%':>9}  {'50%':>9}  {'60%':>9}  {'Avg ratio':>10}")
print("  " + "-" * 66)

for ds in DATASETS:
    if ds not in our_results:
        print(f"  {ds:<12}  (not yet run)")
        continue
    ratios = []
    cells  = []
    for pct in PERCENTAGES:
        our = our_results[ds][pct]["rmse"]
        pap = TABLE4_RMSE[ds][pct]["MIGA"]
        r   = our / pap
        ratios.append(r)
        label = "★" if r <= 1.0 else f"{r:.2f}x"
        cells.append(f"{label:>9}")
    avg = sum(ratios) / len(ratios)
    avg_label = "★" if avg <= 1.0 else f"{avg:.2f}x"
    print(f"  {ds:<12}  {'  '.join(cells)}  {avg_label:>10}")

print()
print("  ★ = beats paper")
print()

print("=== MAD: Our MIGA / Paper MIGA ===")
print()
print(f"  {'Dataset':<12}  {'30%':>9}  {'40%':>9}  {'50%':>9}  {'60%':>9}  {'Avg ratio':>10}")
print("  " + "-" * 66)

for ds in DATASETS:
    if ds not in our_results:
        print(f"  {ds:<12}  (not yet run)")
        continue
    ratios = []
    cells  = []
    for pct in PERCENTAGES:
        our = our_results[ds][pct]["mad"]
        pap = TABLE5_MAD[ds][pct]["MIGA"]
        r   = our / pap
        ratios.append(r)
        label = "★" if r <= 1.0 else f"{r:.2f}x"
        cells.append(f"{label:>9}")
    avg = sum(ratios) / len(ratios)
    avg_label = "★" if avg <= 1.0 else f"{avg:.2f}x"
    print(f"  {ds:<12}  {'  '.join(cells)}  {avg_label:>10}")

print()
print("  ★ = beats paper")
print()

print("=== CoD: Our MIGA / Paper MIGA  (higher CoD is better; ratio > 1 = beats paper) ===")
print()
print(f"  {'Dataset':<12}  {'30%':>9}  {'40%':>9}  {'50%':>9}  {'60%':>9}  {'Avg ratio':>10}")
print("  " + "-" * 66)

for ds in DATASETS:
    if ds not in our_results:
        print(f"  {ds:<12}  (not yet run)")
        continue
    ratios = []
    cells  = []
    for pct in PERCENTAGES:
        our = our_results[ds][pct]["cod"]
        pap = TABLE6_COD[ds][pct]["MIGA"]
        # Ratio is only meaningful when both values are positive
        r = our / pap if (pap > 1e-6 and our > 0) else float("nan")
        ratios.append(r)
        if our >= pap:
            label = "★"
        elif r == r:  # valid ratio
            label = f"{r:.2f}x"
        else:
            # Negative CoD or undefined — show delta instead
            label = f"{our-pap:+.2f}"
        cells.append(f"{label:>9}")
    valid = [r for r in ratios if r == r]
    avg = sum(valid) / len(valid) if valid else float("nan")
    avg_label = "★" if avg >= 1.0 else (f"{avg:.2f}x" if avg == avg else "n/a")
    print(f"  {ds:<12}  {'  '.join(cells)}  {avg_label:>10}")

print()
print("  ★ = meets or beats paper   (CoD: higher = better, ratio > 1 is good)")

## 7. Heat-map: Our MIGA RMSE vs. Paper

In [ ]:
available_ds = [d for d in DATASETS if d in our_results]
if not available_ds:
    print("No results available yet. Run notebooks 02–08 first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, len(available_ds) * 0.8 + 2))

    for ax, (table, title) in zip(axes, [
        (TABLE4_RMSE, "RMSE (lower=better)"),
        (TABLE5_MAD,  "MAD  (lower=better)"),
        (TABLE6_COD,  "CoD  (higher=better)"),
    ]):
        key = "rmse" if "RMSE" in title else ("mad" if "MAD" in title else "cod")
        data_our = np.array([[our_results[d][p][key] for p in PERCENTAGES] for d in available_ds])
        data_pap = np.array([[table[d][p]["MIGA"]    for p in PERCENTAGES] for d in available_ds])
        delta = data_our - data_pap

        im = ax.imshow(delta, cmap="RdYlGn_r" if key != "cod" else "RdYlGn",
                       aspect="auto", vmin=-0.1, vmax=0.1)
        ax.set_xticks(range(4))
        ax.set_xticklabels([f"{p}%" for p in PERCENTAGES])
        ax.set_yticks(range(len(available_ds)))
        ax.set_yticklabels(available_ds)
        ax.set_title(f"Δ {title}\n(Ours − Paper)", fontsize=10)
        for i in range(len(available_ds)):
            for j in range(4):
                ax.text(j, i, f"{delta[i,j]:+.3f}", ha="center", va="center", fontsize=7)
        plt.colorbar(im, ax=ax, fraction=0.04)

    plt.suptitle("MIGA Reimplementation vs. Paper (green = better than paper)", fontsize=12)
    plt.tight_layout()
    plt.savefig("../results/09_heatmap_comparison.png", dpi=120, bbox_inches="tight")
    plt.show()

## 8. Full Table 4 — RMSE All Methods (Our MIGA replaces Paper MIGA)

In [ ]:
import pandas as pd
from miga.paper_results import METHODS

for ds in available_ds:
    rows = []
    for pct in PERCENTAGES:
        row = {"pct": f"{pct}%", "OurMIGA": f"{our_results[ds][pct]['rmse']:.4f}"}
        for m in METHODS:
            row[m] = f"{TABLE4_RMSE[ds][pct].get(m, float('nan')):.4f}"
        rows.append(row)
    df = pd.DataFrame(rows).set_index("pct")
    print(f"\n{ds} — RMSE (Table 4):")
    display(df)